In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append('/iopsstor/scratch/cscs/xyixuan/benchmark-image-tokenzier/SelftokTokenizer')
sys.path.append('/iopsstor/scratch/cscs/xyixuan/benchmark-image-tokenzier/SelftokTokenizer/mimogpt')

In [3]:
import argparse
from SelftokTokenizer.mimogpt.infer.infer_utils import parse_args_from_yaml
from torchvision import transforms
from PIL import Image
import torch
import numpy as np
from mimogpt.infer.SelftokPipeline import SelftokPipeline
from mimogpt.infer.SelftokPipeline import NormalizeToTensor
from torchvision.utils import save_image

[2025-06-26 12:45:34,979] [INFO] [real_accelerator.py:254:get_accelerator] Setting ds_accelerator to cuda (auto detect)


df: /users/xyixuan/.triton/autotune: No such file or directory


/users/xyixuan/miniconda3_x86/envs/selftok/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/users/xyixuan/miniconda3_x86/envs/selftok/compiler_compat/ld: warning: librt.so.1, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/users/xyixuan/miniconda3_x86/envs/selftok/compiler_compat/ld: warning: libpthread.so.0, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/users/xyixuan/miniconda3_x86/envs/selftok/compiler_compat/ld: warning: libstdc++.so.6, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/users/xyixuan/miniconda3_x86/envs/selftok/compiler_compat/ld: warning: libm.so.6, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/users/xyixuan/miniconda3_x86/envs/selftok/compiler_compat/ld: /usr/local/cuda/lib64/libcufile.so: undefined reference to `std::runtime_error::~r

['/users/xyixuan/miniconda3_x86/envs/selftok/lib/python310.zip', '/users/xyixuan/miniconda3_x86/envs/selftok/lib/python3.10', '/users/xyixuan/miniconda3_x86/envs/selftok/lib/python3.10/lib-dynload', '', '/users/xyixuan/miniconda3_x86/envs/selftok/lib/python3.10/site-packages', '/iopsstor/scratch/cscs/xyixuan/benchmark-image-tokenzier/SelftokTokenizer', '/iopsstor/scratch/cscs/xyixuan/benchmark-image-tokenzier/SelftokTokenizer/mimogpt', '/users/xyixuan/miniconda3_x86/envs/selftok/lib/python3.10/site-packages/setuptools/_vendor', '/tmp/tmpuy_gl_nc']
# Ascend didn't support decord, skipped
[2025-06-26 12:45:46,395][   sd3_utils.py][line:  54][    INFO] PyTorch version 2.5.0 available.
Using gradient checkpointing...


/iopsstor/scratch/cscs/xyixuan/benchmark-image-tokenzier/SelftokTokenizer/mimogpt/models/selftok/vector_quantize_pytorch.py:525: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)


In [ ]:
# Configuration - edit these paths as needed
CONFIG = {
    'yml_path': "./configs/res256/256-eval.yml",
    'pretrained': "path/to/your/tokenizer_512_ckpt.pth",
    'sd3_pretrained': "path/to/your/models--stabilityai--stable-diffusion-3-medium-diffusers",
    'data_size': 256
}

cfg = parse_args_from_yaml(CONFIG['yml_path'])
model = SelftokPipeline(cfg=cfg, ckpt_path=CONFIG['pretrained'], 
                       sd3_path=CONFIG['sd3_pretrained'], 
                       datasize=CONFIG['data_size'], device='cuda')

img_transform = transforms.Compose([
    transforms.Resize(CONFIG['data_size']),
    transforms.CenterCrop(CONFIG['data_size']),
    NormalizeToTensor(),
])

image_paths = ['./test.jpg']
images = [img_transform(Image.open(p)) for p in image_paths]
images = torch.stack(images).to('cuda')

tokens = model.encoding(images, device='cuda')
np.save('./token.npy', tokens.detach().cpu().numpy())
tokens = np.load('./token.npy')